# Instamart Customer Intelligence

An end-to-end, reproducible walkthrough of the synthetic Instamart capstone: data auditing and cleaning, SQL and pandas analysis, RFM segmentation, privacy-preserving ticket triage, and a retrieval-grounded policy/refund agent.

> **Reproducibility note:** The checked-in deterministic generator is the source of truth. Its data does not match the assignment's numeric reference checkpoints (for example, it generates 13,846 raw order rows and 13,084 distinct order IDs). This notebook reports results from the generated data rather than changing data to force reference values.

Run the notebook from top to bottom. Project scripts write files under `data/`, `outputs/`, and `state/`; rerunning the generation and pipeline cells overwrites generated files in those locations.

## 1. Locate the project and prepare the environment

The notebook searches the current directory and its parents for the repository markers. Install the dependencies from `requirements.txt` in the selected kernel environment before continuing if imports fail.

In [ ]:
from pathlib import Path
import json
import sqlite3
import subprocess
import sys

def find_project_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "README.md").is_file() and (candidate / "generate_data.py").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find the project root. Open this notebook from the repository "
        "or set ROOT to the repository directory manually."
    )

ROOT = find_project_root()
print(f"Project root: {ROOT}")
print(f"Python executable: {sys.executable}")

def run_project_script(*arguments: str) -> None:
    command = [sys.executable, *arguments]
    print("\n$", " ".join(command))
    subprocess.run(command, cwd=ROOT, check=True)


In [ ]:
import numpy as np
import pandas as pd

print(f"pandas {pd.__version__} | numpy {np.__version__}")
print("Requirements:")
print((ROOT / "requirements.txt").read_text(encoding="utf-8"))


## 2. Inspect the project inputs

Raw CSV exports and policy documents are generated by `generate_data.py`. Clean exports and the SQLite database are created by the pipeline.

In [ ]:
raw_dir = ROOT / "data" / "raw"
for path in sorted(raw_dir.glob("*.csv")):
    frame = pd.read_csv(path, keep_default_na=False)
    print(f"{path.relative_to(ROOT)}: {frame.shape[0]:,} rows x {frame.shape[1]} columns")
    display(frame.head(3))

print("Policy documents:")
for path in sorted((ROOT / "data" / "policies").glob("*.txt")):
    print(f"- {path.name}")


## 3. Regenerate source data and run the core pipeline

The generator is deterministic. The core pipeline audits raw exports, standardizes customers/products/orders, writes clean CSVs and SQLite tables, validates core invariants, executes its SQL analyses, and creates initial support/policy outputs. Later sections run the dedicated Part C and Part D modules after segmentation so their final outputs use the complete workflows.

In [ ]:
run_project_script("generate_data.py")
run_project_script("src/instamart_pipeline.py")


## 4. Validate cleaned data

The validator checks data-quality rules and consistency between clean CSVs and the SQLite database. It does not enforce incompatible assignment reference counts.

In [ ]:
run_project_script("src/validate_data.py")

clean_dir = ROOT / "data" / "clean"
customers = pd.read_csv(clean_dir / "customers.csv", parse_dates=["signup_date"])
products = pd.read_csv(clean_dir / "products.csv")
orders = pd.read_csv(clean_dir / "orders.csv", parse_dates=["order_ts"])

for name, frame in (("customers", customers), ("products", products), ("orders", orders)):
    print(f"{name}: {frame.shape[0]:,} rows; {frame.shape[1]} columns")
    display(frame.head(3))

display(orders.groupby("status", dropna=False).size().rename("rows").to_frame())


## 5. Part A - SQL and pandas analysis

The project contains seven SQL examples: filtering with `HAVING`, multi-table joins, a scalar subquery, `LEFT JOIN`, a CTE, ranking window functions, and cumulative daily revenue. The analysis script also cross-checks results with pandas.

In [ ]:
run_project_script("src/run_analysis.py")

database_path = ROOT / "data" / "instamart.db"
sql_text = (ROOT / "sql" / "queries.sql").read_text(encoding="utf-8")
query_blocks = sql_text.split("-- Q")[1:]
queries = [
    (block.splitlines()[0].strip(), block.split("\n", 1)[1].strip())
    for block in query_blocks
    if block.strip()
]
print(f"Loaded {len(queries)} named SQL queries")

with sqlite3.connect(database_path) as connection:
    table_names = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name",
        connection,
    )["name"].tolist()
    sql_results = {
        title: pd.read_sql_query(query, connection)
        for title, query in queries
    }

print("SQLite tables:", table_names)
for title, result in sql_results.items():
    print(f"\n{title}: {len(result):,} rows")
    display(result.head(5))

revenue = sql_results["2: three-table join"]["revenue"].sum()
print(f"Completed-order revenue across city/category groups: INR {revenue:,.2f}")


## 6. Part B - RFM segmentation and lapse-data check

RFM features are recency, completed-order frequency, and monetary spend. The script tests K-means cluster counts, profiles three selected groups, and writes `outputs/customer_segments.csv`. Its lapse-label window is September 2025. The supplied synthetic orders end in 2024, so the window has no completed orders: every cohort row is labeled lapsed. The workflow reports this single-class limitation and correctly stops instead of fitting an invalid two-class model.

In [ ]:
run_project_script("src/run_part_b.py")

segments_path = ROOT / "outputs" / "customer_segments.csv"
segments = pd.read_csv(segments_path)
print(f"Segments: {len(segments):,} customers")
display(
    segments.groupby(["segment_id", "segment_name"], as_index=False)
    .agg(
        customers=("customer_id", "size"),
        mean_recency_days=("recency", "mean"),
        mean_frequency=("frequency", "mean"),
        mean_monetary_inr=("monetary", "mean"),
        premium_customers=("is_premium", "sum"),
    )
    .sort_values("segment_id")
)
display(segments.head())


### Segmentation caveat

RFM is calculated as of 2025-10-01, but the latest completed orders in the generated dataset are from 2024. The Premium indicator is an illustrative whole-window segment, not a leakage-safe retention evaluation. Lapse modeling, class balancing, and retention-list ranking require source data with both lapsed and non-lapsed labels in the target window.

## 7. Part C - Privacy-preserving ticket triage

The dedicated triage workflow redacts email, phone, and order-ID patterns; rejects empty and overlong tickets before model calls; retries transient mock failures; validates JSON or a single allowed category; and routes invalid cases to human review. It compares zero-shot/few-shot prompting and writes a token-budget demonstration plus the final few-shot result. The LLM is an offline deterministic mock, not an external service.

In [ ]:
run_project_script("-m", "src.triage_tickets")

triage_path = ROOT / "outputs" / "triage_results.csv"
triage = pd.read_csv(triage_path, keep_default_na=False)
print(f"Final triage result rows: {len(triage):,}")
display(triage.head(10))
for column in ("status", "category", "route"):
    if column in triage.columns:
        print(f"\n{column} counts:")
        display(triage[column].value_counts(dropna=False).rename("tickets").to_frame())

budget_demo_path = ROOT / "outputs" / "triage_budget_demo.csv"
if budget_demo_path.is_file():
    budget_demo = pd.read_csv(budget_demo_path, keep_default_na=False)
    print(f"Budget demo rows: {len(budget_demo):,}")
    if "status" in budget_demo.columns:
        display(budget_demo["status"].value_counts().rename("tickets").to_frame())


## 8. Part D - Policy retrieval and safeguarded refund agent

Policy documents are split into overlapping word chunks and searched with TF-IDF cosine similarity. The evaluation compares the default 10-word overlap with no overlap. The refund agent uses database-backed order lookup, policy search, refund calculations, eligibility checks, automatic-refund limits, human escalation, token/time limits, and checkpoint-based crash/resume handling. This is a simulation using the supplied local mock.

In [ ]:
run_project_script("-m", "src.policy_agent")

agent_runs_path = ROOT / "outputs" / "agent_runs.jsonl"
agent_runs = [
    json.loads(line)
    for line in agent_runs_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
agent_summary = pd.DataFrame(
    [
        {
            "scenario_id": run["scenario_id"],
            "status": run["status"],
            "llm_calls": run["llm_calls"],
            "tokens_used": run["tokens_used"],
            "tool_steps": len(run["steps"]),
        }
        for run in agent_runs
    ]
)
display(agent_summary)

ledger_path = ROOT / "outputs" / "refund_ledger.jsonl"
ledger = [
    json.loads(line)
    for line in ledger_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
print(f"Validated refund/escalation ledger entries: {len(ledger)}")
if ledger:
    display(pd.DataFrame(ledger).head())

resume_checkpoint = ROOT / "state" / "crash_resume_S1.json"
print(f"Crash/resume checkpoint exists: {resume_checkpoint.is_file()}")


## 9. Generated deliverables and project limitations

The main deliverables include cleaned CSVs and SQLite tables, SQL query results, customer segments, triage results, policy-agent runs, the refund ledger, and agent checkpoints. The dataset is synthetic and its date range does not support the requested September 2025 lapse modeling. Triage and policy behavior is exercised against a local mock, so it should not be interpreted as production model performance or a live financial system.

In [ ]:
deliverables = [
    "data/clean/customers.csv",
    "data/clean/products.csv",
    "data/clean/orders.csv",
    "data/instamart.db",
    "sql/queries.sql",
    "outputs/customer_segments.csv",
    "outputs/triage_results.csv",
    "outputs/triage_budget_demo.csv",
    "outputs/agent_runs.jsonl",
    "outputs/refund_ledger.jsonl",
    "state/crash_resume_S1.json",
]
deliverable_status = pd.DataFrame(
    {
        "path": deliverables,
        "exists": [(ROOT / relative_path).is_file() for relative_path in deliverables],
    }
)
display(deliverable_status)
assert deliverable_status["exists"].all(), "One or more expected project deliverables are missing."
print("End-to-end project walkthrough complete.")
